### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [12]:
import os
from langchain.chat_models import init_chat_model
import dotenv

dotenv.load_dotenv()  # Load environment variables from .env file
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")  # Set your Groq API key as an environment variable

model = init_chat_model("groq:llama-3.1-8b-instant")   # qwen2.5-mini
response = model.invoke("Write me a poem about a cat in 500 words")

In [13]:
from langchain.tools import tool

@tool # @tool("get_current_weather", return_direct=True)
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_current_weather])

In [14]:
response = model_with_tools.invoke("What is the current weather in New York?")
print(response) ## llm models can understand the tool, when and which tool need to be called.
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Input : {tool_call['args']}")



content='' additional_kwargs={'tool_calls': [{'id': 'eq0rnmqkc', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 227, 'total_tokens': 243, 'completion_time': 0.039727171, 'completion_tokens_details': None, 'prompt_time': 0.016198423, 'prompt_tokens_details': None, 'queue_time': 0.054220232, 'total_time': 0.055925594}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019dfe04-aced-73e3-8bb7-336cd09697cd-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'eq0rnmqkc', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 227, 'output_tokens': 16, 'total_tokens': 243}
Tool : get_current_weather
Input : {'location': 'New York'}


### Tool Execution Loop

In [15]:
# Step 1 : model generates tool call
messages = [{"role": "user", "content": "What is the current weather in New York?"}]
ai_response = model_with_tools.invoke(messages)
messages.append(ai_response)  # Model's response to the user

# Step 2 : tool call
for tool_call in ai_response.tool_calls:
    print(f"Tool : {tool_call}")
    # Execute the tool with the generated arguments
    tool_result = get_current_weather.invoke(tool_call)
    print(f"Tool Result : {tool_result}")
    messages.append(tool_result)

# Step 3 : Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)



Tool : {'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'w8jdwt3xn', 'type': 'tool_call'}
Tool Result : content='The current weather in New York is sunny with a temperature of 25°C.' name='get_current_weather' tool_call_id='w8jdwt3xn'
content='' additional_kwargs={'tool_calls': [{'id': '2qe7cgh08', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 269, 'total_tokens': 285, 'completion_time': 0.016341168, 'completion_tokens_details': None, 'prompt_time': 0.016367549, 'prompt_tokens_details': None, 'queue_time': 0.053074981, 'total_time': 0.032708717}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019dfe04-aeab-7203-9ac6-f2ad2a0dafc4-0' tool_calls=[{'name': 'get_current_weather', 'args': {

In [16]:
messages

[{'role': 'user', 'content': 'What is the current weather in New York?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'w8jdwt3xn', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 227, 'total_tokens': 243, 'completion_time': 0.037690414, 'completion_tokens_details': None, 'prompt_time': 0.035180507, 'prompt_tokens_details': None, 'queue_time': 0.053893372, 'total_time': 0.072870921}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dfe04-ae03-7251-b998-90b917c6f8dd-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'w8jdwt3xn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 227, 'output_tokens': 16, 'total_tokens': 243})